### Regression (RNN, GRU, LSTM, Mamba) Models

Stats Features Only (Regression) -- Training

In [1]:
# === Try more

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from mamba_ssm import Mamba
import math
import json

# ========== Device ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet3"
)

target = "OSI_V2_12th_avg"

base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

X, y, groups = [], [], []

for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target]):
        X.append(sequence)
        y.append(row[target])
        groups.append(row['ResearchID'] if 'ResearchID' in row else idx)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

feature_dim = X.shape[2]

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y_scaled, groups, test_size=0.2, random_state=42
)

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim) for _ in range(num_layers)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

# ========== Train and Evaluate ==========
def get_loader(X, y, batch_size):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

def run_grid_search():
    models = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]
    hidden_dims = [16, 32, 64, 128]
    dropouts = [0.0, 0.2]
    num_layers_list = [1, 2, 3]
    batch_sizes = [16, 32]
    lrs = [0.001, 0.01]
    weight_decays = [0, 1e-4]
    optimizers = ["Adam", "SGD"]
    epochs = 200
    results = {"cv_predictions": [], "summary": []}
    gkf = GroupKFold(n_splits=5)

    for model_type in models:
        for hidden_dim in hidden_dims:
            for dropout in dropouts:
                for num_layers in num_layers_list:
                    for batch_size in batch_sizes:
                        for lr in lrs:
                            for wd in weight_decays:
                                for opt_name in optimizers:
                                    rmse_list, mae_list = [], []
                                    used_epochs_all = []
                                    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                                        if model_type == "Mamba":
                                            model = MambaModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        elif model_type == "Transformer":
                                            model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        else:
                                            model = RNNModel(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)
                                        optimizer_cls = torch.optim.Adam if opt_name == "Adam" else torch.optim.SGD
                                        optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
                                        criterion = nn.MSELoss()
                                        loader = get_loader(X_train[train_idx], y_train[train_idx], batch_size)

                                        best_loss = float('inf')
                                        wait = 0
                                        used_epochs = 0
                                        for epoch in range(epochs):
                                            model.train()
                                            epoch_losses = []
                                            for xb, yb in loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                epoch_losses.append(loss.item())
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()
                                            avg_loss = np.mean(epoch_losses)
                                            scheduler.step(avg_loss)
                                            used_epochs += 1
                                            if avg_loss < best_loss:
                                                best_loss = avg_loss
                                                wait = 0
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break
                                        used_epochs_all.append(used_epochs)

                                        model.eval()
                                        val_x = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
                                        preds = model(val_x).squeeze().detach().cpu().numpy()
                                        preds = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
                                        y_true = scaler_y.inverse_transform(y_train[val_idx].reshape(-1, 1)).flatten()

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": opt_name,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_pred": preds.tolist()
                                        })

                                        rmse = np.sqrt(mean_squared_error(y_true, preds))
                                        mae = mean_absolute_error(y_true, preds)
                                        rmse_list.append(rmse)
                                        mae_list.append(mae)

                                    results["summary"].append({
                                        "model": model_type, "hidden_dim": hidden_dim, "dropout": dropout,
                                        "num_layers": num_layers, "batch_size": batch_size, "lr": lr,
                                        "weight_decay": wd, "optimizer": opt_name, "epochs": epochs,
                                        "rmse_mean": np.mean(rmse_list), "rmse_std": np.std(rmse_list),
                                        "mae_mean": np.mean(mae_list), "mae_std": np.std(mae_list),
                                        "used_epochs_mean": np.mean(used_epochs_all), "used_epochs_std": np.std(used_epochs_all)
                                    })
                                    print(f"{model_type} HD={hidden_dim} DO={dropout} NL={num_layers} EP={epochs} OPT={opt_name} => RMSE: {np.mean(rmse_list):.4f} ± {np.std(rmse_list):.4f}, MAE: {np.mean(mae_list):.4f} ± {np.std(mae_list):.4f}")
    return pd.DataFrame(results["summary"]), results["cv_predictions"]

# # ========== Run ==========
# results_df, cv_predictions = run_grid_search()
# results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(Stats)_5.csv", index=False)
# print("✅ Grid search completed and results saved.")

# with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_cv_predictions(Stats)_5.json", "w") as f:
#     json.dump(cv_predictions, f)
# print(f"✅ CV predictions saved: {len(cv_predictions)} records.")


Using device: cuda


Stats Features Only (Regression) -- Test

In [2]:
# Load summary CSV
df_results = pd.read_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(Stats)_5.csv")
best_per_model = df_results.sort_values("rmse_mean").groupby("model").first().reset_index()
print("\nBest configuration per model:\n", best_per_model)

results = []
predictions = []

for _, row in best_per_model.iterrows():
    model_type = row["model"]
    hidden_dim = int(row["hidden_dim"])
    dropout = float(row["dropout"])
    num_layers = int(row["num_layers"])
    epochs = int(row["epochs"])
    batch_size = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr = float(row["lr"])
    wd = float(row["weight_decay"])
    input_dim = X_train.shape[2]

    # Model init
    if model_type == "Mamba":
        model = MambaModel(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        model = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        model = RNNModel(input_dim, hidden_dim, model_type, num_layers, dropout)
    model = model.to(device)

    # Optimizer & training setup
    optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    loss_fn = nn.MSELoss()
    train_loader = get_loader(X_train, y_train, batch_size)

    best_loss = float('inf')
    wait = 0
    used_epochs = 0

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        if len(epoch_losses) == 0:
            continue

        avg_loss = np.mean(epoch_losses)
        scheduler.step(avg_loss)
        used_epochs += 1

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state_dict = model.state_dict()
        else:
            wait += 1
            if wait >= 10:
                print(f"⏹️ Early stopping at epoch {used_epochs} for model {model_type}")
                break

    # Restore best model
    model.load_state_dict(best_state_dict)

    # Evaluate on test set
    model.eval()
    preds = []
    with torch.no_grad():
        for xb_batch in get_loader(X_test, y_test, batch_size=batch_size):
            xb = xb_batch[0].to(device)
            pred = model(xb).cpu().numpy().flatten()
            preds.append(pred)
    y_pred_scaled = np.concatenate(preds)

    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    results.append({
        "model": model_type,
        "rmse_test": rmse,
        "mae_test": mae,
        "used_epochs": used_epochs
    })

    predictions.append({
        "model": model_type,
        "y_true": y_true.tolist(),
        "y_pred": y_pred.tolist()
    })

# Save test performance
results_df = pd.DataFrame(results)
results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_results(Stats)_5.csv", index=False)
print("✅ Test performance of best models saved.")
print(results_df)

# Save test predictions
with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_predictions(Stats)_5.json", "w") as f:
    json.dump(predictions, f)
print("✅ Test predictions saved.")



Best configuration per model:
          model  hidden_dim  dropout  num_layers  batch_size     lr  \
0          GRU          16      0.2           3          16  0.010   
1         LSTM          32      0.2           2          16  0.010   
2        Mamba         128      0.2           2          32  0.001   
3          RNN          64      0.2           3          16  0.001   
4  Transformer          16      0.0           3          32  0.001   

   weight_decay optimizer  epochs  rmse_mean  rmse_std  mae_mean   mae_std  \
0        0.0001       SGD     200   2.737983  0.668906  1.767512  0.309529   
1        0.0001       SGD     200   2.735344  0.685274  1.771290  0.323469   
2        0.0001      Adam     200   2.804109  0.590889  1.854306  0.306819   
3        0.0000       SGD     200   2.723184  0.656115  1.737787  0.277075   
4        0.0000      Adam     200   2.748041  0.660847  1.877310  0.333341   

   used_epochs_mean  used_epochs_std  
0              43.0         5.656854  


CNN Features (ALL) Only (Regression) -- Training

In [5]:
# === Try more

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from mamba_ssm import Mamba
import math
import json

# ========== Device ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet3"
)

target = "OSI_V2_12th_avg"

tw_features = {tw: [f"f{i}_TW{tw}" for i in range(1, 257)] for tw in range(1, 7)}

X, y, groups = [], [], []
for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target]):
        X.append(sequence)
        y.append(row[target])
        groups.append(row['ResearchID'] if 'ResearchID' in row else idx)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

feature_dim = X.shape[2]

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y_scaled, groups, test_size=0.2, random_state=42
)

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim) for _ in range(num_layers)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

# ========== Train and Evaluate ==========
def get_loader(X, y, batch_size):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

def run_grid_search():
    models = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]
    hidden_dims = [16, 32, 64, 128]
    dropouts = [0.0, 0.2]
    num_layers_list = [1, 2, 3]
    batch_sizes = [16, 32]
    lrs = [0.001, 0.01]
    weight_decays = [0, 1e-4]
    optimizers = ["Adam", "SGD"]
    epochs = 200
    results = {"cv_predictions": [], "summary": []}
    gkf = GroupKFold(n_splits=5)

    for model_type in models:
        for hidden_dim in hidden_dims:
            for dropout in dropouts:
                for num_layers in num_layers_list:
                    for batch_size in batch_sizes:
                        for lr in lrs:
                            for wd in weight_decays:
                                for opt_name in optimizers:
                                    rmse_list, mae_list = [], []
                                    used_epochs_all = []
                                    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                                        if model_type == "Mamba":
                                            model = MambaModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        elif model_type == "Transformer":
                                            model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        else:
                                            model = RNNModel(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)
                                        optimizer_cls = torch.optim.Adam if opt_name == "Adam" else torch.optim.SGD
                                        optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
                                        criterion = nn.MSELoss()
                                        loader = get_loader(X_train[train_idx], y_train[train_idx], batch_size)

                                        best_loss = float('inf')
                                        wait = 0
                                        used_epochs = 0
                                        for epoch in range(epochs):
                                            model.train()
                                            epoch_losses = []
                                            for xb, yb in loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                epoch_losses.append(loss.item())
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()
                                            avg_loss = np.mean(epoch_losses)
                                            scheduler.step(avg_loss)
                                            used_epochs += 1
                                            if avg_loss < best_loss:
                                                best_loss = avg_loss
                                                wait = 0
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break
                                        used_epochs_all.append(used_epochs)

                                        model.eval()
                                        val_x = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
                                        preds = model(val_x).squeeze().detach().cpu().numpy()
                                        preds = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
                                        y_true = scaler_y.inverse_transform(y_train[val_idx].reshape(-1, 1)).flatten()

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": opt_name,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_pred": preds.tolist()
                                        })

                                        rmse = np.sqrt(mean_squared_error(y_true, preds))
                                        mae = mean_absolute_error(y_true, preds)
                                        rmse_list.append(rmse)
                                        mae_list.append(mae)

                                    results["summary"].append({
                                        "model": model_type, "hidden_dim": hidden_dim, "dropout": dropout,
                                        "num_layers": num_layers, "batch_size": batch_size, "lr": lr,
                                        "weight_decay": wd, "optimizer": opt_name, "epochs": epochs,
                                        "rmse_mean": np.mean(rmse_list), "rmse_std": np.std(rmse_list),
                                        "mae_mean": np.mean(mae_list), "mae_std": np.std(mae_list),
                                        "used_epochs_mean": np.mean(used_epochs_all), "used_epochs_std": np.std(used_epochs_all)
                                    })
                                    print(f"{model_type} HD={hidden_dim} DO={dropout} NL={num_layers} EP={epochs} OPT={opt_name} => RMSE: {np.mean(rmse_list):.4f} ± {np.std(rmse_list):.4f}, MAE: {np.mean(mae_list):.4f} ± {np.std(mae_list):.4f}")
    return pd.DataFrame(results["summary"]), results["cv_predictions"]

# # ========== Run ==========
# results_df, cv_predictions = run_grid_search()
# results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(CNN)_5.csv", index=False)
# print("✅ Grid search completed and results saved.")

# with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_cv_predictions(CNN)_5.json", "w") as f:
#     json.dump(cv_predictions, f)
# print(f"✅ CV predictions saved: {len(cv_predictions)} records.")


Using device: cuda


CNN Features (ALL) Only (Regression) -- Test

In [6]:
# Load summary CSV
df_results = pd.read_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(CNN)_5.csv")
best_per_model = df_results.sort_values("rmse_mean").groupby("model").first().reset_index()
print("\nBest configuration per model:\n", best_per_model)

results = []
predictions = []

for _, row in best_per_model.iterrows():
    model_type = row["model"]
    hidden_dim = int(row["hidden_dim"])
    dropout = float(row["dropout"])
    num_layers = int(row["num_layers"])
    epochs = int(row["epochs"])
    batch_size = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr = float(row["lr"])
    wd = float(row["weight_decay"])
    input_dim = X_train.shape[2]

    # Model init
    if model_type == "Mamba":
        model = MambaModel(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        model = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        model = RNNModel(input_dim, hidden_dim, model_type, num_layers, dropout)
    model = model.to(device)

    # Optimizer & training setup
    optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    loss_fn = nn.MSELoss()
    train_loader = get_loader(X_train, y_train, batch_size)

    best_loss = float('inf')
    wait = 0
    used_epochs = 0

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        if len(epoch_losses) == 0:
            continue

        avg_loss = np.mean(epoch_losses)
        scheduler.step(avg_loss)
        used_epochs += 1

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state_dict = model.state_dict()
        else:
            wait += 1
            if wait >= 10:
                print(f"⏹️ Early stopping at epoch {used_epochs} for model {model_type}")
                break

    # Restore best model
    model.load_state_dict(best_state_dict)

    # Evaluate on test set
    model.eval()
    preds = []
    with torch.no_grad():
        for xb_batch in get_loader(X_test, y_test, batch_size=batch_size):
            xb = xb_batch[0].to(device)
            pred = model(xb).cpu().numpy().flatten()
            preds.append(pred)
    y_pred_scaled = np.concatenate(preds)

    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    results.append({
        "model": model_type,
        "rmse_test": rmse,
        "mae_test": mae,
        "used_epochs": used_epochs
    })

    predictions.append({
        "model": model_type,
        "y_true": y_true.tolist(),
        "y_pred": y_pred.tolist()
    })

# Save test performance
results_df = pd.DataFrame(results)
results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_results(CNN)_5.csv", index=False)
print("✅ Test performance of best models saved.")
print(results_df)

# Save test predictions
with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_predictions(CNN)_5.json", "w") as f:
    json.dump(predictions, f)
print("✅ Test predictions saved.")



Best configuration per model:
          model  hidden_dim  dropout  num_layers  batch_size     lr  \
0          GRU          16      0.0           3          16  0.001   
1         LSTM         128      0.2           2          16  0.001   
2        Mamba          16      0.2           2          16  0.010   
3          RNN         128      0.0           3          16  0.001   
4  Transformer          32      0.2           3          32  0.010   

   weight_decay optimizer  epochs  rmse_mean  rmse_std  mae_mean   mae_std  \
0        0.0001       SGD     200   2.564731  0.508459  1.732886  0.195855   
1        0.0001       SGD     200   2.578934  0.465811  1.744729  0.150119   
2        0.0000       SGD     200   2.649644  0.404389  1.838225  0.176167   
3        0.0000       SGD     200   2.579011  0.463853  1.740115  0.167972   
4        0.0000       SGD     200   2.627604  0.462120  1.792895  0.175975   

   used_epochs_mean  used_epochs_std  
0             115.6        23.087659  


Stats + CNN Features (Regression) -- Training

In [7]:
# === Try more

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from mamba_ssm import Mamba
import math
import json

# ========== Device ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet3"
)

target = "OSI_V2_12th_avg"

base_template = ["OSI_mean_TW{}", "OSI_std_TW{}"]

tw_features = {
    tw: [f.format(tw) for f in base_template] + [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

X, y, groups = [], [], []
for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target]):
        X.append(sequence)
        y.append(row[target])
        groups.append(row['ResearchID'] if 'ResearchID' in row else idx)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

feature_dim = X.shape[2]

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y_scaled, groups, test_size=0.2, random_state=42
)

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim) for _ in range(num_layers)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

# ========== Train and Evaluate ==========
def get_loader(X, y, batch_size):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

def run_grid_search():
    models = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]
    hidden_dims = [16, 32, 64, 128]
    dropouts = [0.0, 0.2]
    num_layers_list = [1, 2, 3]
    batch_sizes = [16, 32]
    lrs = [0.001, 0.01]
    weight_decays = [0, 1e-4]
    optimizers = ["Adam", "SGD"]
    epochs = 200
    results = {"cv_predictions": [], "summary": []}
    gkf = GroupKFold(n_splits=5)

    for model_type in models:
        for hidden_dim in hidden_dims:
            for dropout in dropouts:
                for num_layers in num_layers_list:
                    for batch_size in batch_sizes:
                        for lr in lrs:
                            for wd in weight_decays:
                                for opt_name in optimizers:
                                    rmse_list, mae_list = [], []
                                    used_epochs_all = []
                                    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                                        if model_type == "Mamba":
                                            model = MambaModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        elif model_type == "Transformer":
                                            model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        else:
                                            model = RNNModel(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)
                                        optimizer_cls = torch.optim.Adam if opt_name == "Adam" else torch.optim.SGD
                                        optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
                                        criterion = nn.MSELoss()
                                        loader = get_loader(X_train[train_idx], y_train[train_idx], batch_size)

                                        best_loss = float('inf')
                                        wait = 0
                                        used_epochs = 0
                                        for epoch in range(epochs):
                                            model.train()
                                            epoch_losses = []
                                            for xb, yb in loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                epoch_losses.append(loss.item())
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()
                                            avg_loss = np.mean(epoch_losses)
                                            scheduler.step(avg_loss)
                                            used_epochs += 1
                                            if avg_loss < best_loss:
                                                best_loss = avg_loss
                                                wait = 0
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break
                                        used_epochs_all.append(used_epochs)

                                        model.eval()
                                        val_x = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
                                        preds = model(val_x).squeeze().detach().cpu().numpy()
                                        preds = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
                                        y_true = scaler_y.inverse_transform(y_train[val_idx].reshape(-1, 1)).flatten()

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": opt_name,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_pred": preds.tolist()
                                        })

                                        rmse = np.sqrt(mean_squared_error(y_true, preds))
                                        mae = mean_absolute_error(y_true, preds)
                                        rmse_list.append(rmse)
                                        mae_list.append(mae)

                                    results["summary"].append({
                                        "model": model_type, "hidden_dim": hidden_dim, "dropout": dropout,
                                        "num_layers": num_layers, "batch_size": batch_size, "lr": lr,
                                        "weight_decay": wd, "optimizer": opt_name, "epochs": epochs,
                                        "rmse_mean": np.mean(rmse_list), "rmse_std": np.std(rmse_list),
                                        "mae_mean": np.mean(mae_list), "mae_std": np.std(mae_list),
                                        "used_epochs_mean": np.mean(used_epochs_all), "used_epochs_std": np.std(used_epochs_all)
                                    })
                                    print(f"{model_type} HD={hidden_dim} DO={dropout} NL={num_layers} EP={epochs} OPT={opt_name} => RMSE: {np.mean(rmse_list):.4f} ± {np.std(rmse_list):.4f}, MAE: {np.mean(mae_list):.4f} ± {np.std(mae_list):.4f}")
    return pd.DataFrame(results["summary"]), results["cv_predictions"]

# # ========== Run ==========
# results_df, cv_predictions = run_grid_search()
# results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(Stats_CNN)_5.csv", index=False)
# print("✅ Grid search completed and results saved.")

# with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_cv_predictions(Stats_CNN)_5.json", "w") as f:
#     json.dump(cv_predictions, f)
# print(f"✅ CV predictions saved: {len(cv_predictions)} records.")


Using device: cuda


Stats + CNN Features (Regression) -- Test

In [8]:
# Load summary CSV
df_results = pd.read_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(Stats_CNN)_5.csv")
best_per_model = df_results.sort_values("rmse_mean").groupby("model").first().reset_index()
print("\nBest configuration per model:\n", best_per_model)

results = []
predictions = []

for _, row in best_per_model.iterrows():
    model_type = row["model"]
    hidden_dim = int(row["hidden_dim"])
    dropout = float(row["dropout"])
    num_layers = int(row["num_layers"])
    epochs = int(row["epochs"])
    batch_size = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr = float(row["lr"])
    wd = float(row["weight_decay"])
    input_dim = X_train.shape[2]

    # Model init
    if model_type == "Mamba":
        model = MambaModel(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        model = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        model = RNNModel(input_dim, hidden_dim, model_type, num_layers, dropout)
    model = model.to(device)

    # Optimizer & training setup
    optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    loss_fn = nn.MSELoss()
    train_loader = get_loader(X_train, y_train, batch_size)

    best_loss = float('inf')
    wait = 0
    used_epochs = 0

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        if len(epoch_losses) == 0:
            continue

        avg_loss = np.mean(epoch_losses)
        scheduler.step(avg_loss)
        used_epochs += 1

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state_dict = model.state_dict()
        else:
            wait += 1
            if wait >= 10:
                print(f"⏹️ Early stopping at epoch {used_epochs} for model {model_type}")
                break

    # Restore best model
    model.load_state_dict(best_state_dict)

    # Evaluate on test set
    model.eval()
    preds = []
    with torch.no_grad():
        for xb_batch in get_loader(X_test, y_test, batch_size=batch_size):
            xb = xb_batch[0].to(device)
            pred = model(xb).cpu().numpy().flatten()
            preds.append(pred)
    y_pred_scaled = np.concatenate(preds)

    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    results.append({
        "model": model_type,
        "rmse_test": rmse,
        "mae_test": mae,
        "used_epochs": used_epochs
    })

    predictions.append({
        "model": model_type,
        "y_true": y_true.tolist(),
        "y_pred": y_pred.tolist()
    })

# Save test performance
results_df = pd.DataFrame(results)
results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_results(Stats_CNN)_5.csv", index=False)
print("✅ Test performance of best models saved.")
print(results_df)

# Save test predictions
with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_predictions(Stats_CNN)_5.json", "w") as f:
    json.dump(predictions, f)
print("✅ Test predictions saved.")



Best configuration per model:
          model  hidden_dim  dropout  num_layers  batch_size     lr  \
0          GRU         128      0.2           1          32  0.001   
1         LSTM          64      0.2           3          32  0.010   
2        Mamba          32      0.2           2          16  0.010   
3          RNN          16      0.2           3          16  0.010   
4  Transformer          16      0.2           1          16  0.001   

   weight_decay optimizer  epochs  rmse_mean  rmse_std  mae_mean   mae_std  \
0        0.0001       SGD     200   2.864371  0.557633  1.855999  0.187121   
1        0.0000       SGD     200   2.872805  0.542493  1.873200  0.176393   
2        0.0000      Adam     200   3.031148  0.669852  2.009716  0.317577   
3        0.0000       SGD     200   2.882032  0.581080  1.832811  0.221311   
4        0.0001       SGD     200   2.889240  0.602842  1.959322  0.256426   

   used_epochs_mean  used_epochs_std  
0              81.8        30.805194  


### Classification (RNN, GRU, LSTM, Mamba) Models

Stats Features Only (Classification) -- Training

In [1]:
# === Try more

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from collections import defaultdict
from mamba_ssm import Mamba
import math
import json

# ========== Device ==========
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet3"
)

target_col = "OSI_V2_12th_avg"

base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

X, y, groups = [], [], []

for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target_col]):
        X.append(sequence)
        y.append(1 if row[target_col] >= 7.5 else 0)
        groups.append(row['ResearchID'] if 'ResearchID' in row else idx)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

feature_dim = X.shape[2]

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y, groups, test_size=0.2, random_state=42
)

print(f"Training samples before augmentation: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

print(f"Training samples after augmentation: {len(X_train)}")

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

# ========== Utility ==========
def get_loader(X, y, batch_size):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

def auprc(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3):
# def stratified_sample(y_train, ratio=0.8, pos_ratio=1/3):
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]
    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1
    if len(idx_0) < n0 or len(idx_1) < n1:
        return None, None
    idx = np.concatenate([
        np.random.choice(idx_0, n0, replace=False),
        np.random.choice(idx_1, n1, replace=False)
    ])
    np.random.shuffle(idx)
    return idx, total

# # ========== Train Loop ==========
# epochs = 200
# results = {"cv_predictions": [], "summary": []}
# model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

# for model_type in model_types:
#     for hidden_dim in [64, 128]:
#         for dropout in [0.0, 0.2, 0.4]:
#             for num_layers in [1, 2, 3]:
#                 for batch_size in [16, 32]:
#                     for lr in [0.001, 0.01]:
#                         for wd in [0, 1e-4]:
#                             for optimizer_name in ["Adam", "SGD"]:
#                                 all_aucs, all_auprcs, all_f1s = [], [], []
#                                 used_epochs_all = []
#                                 for repeat in range(10):
#                                     sampled_idx, _ = stratified_sample(y_train)
#                                     if sampled_idx is None:
#                                         print(f"[{model_type}] Skipping due to not enough positive or negative samples.")
#                                         continue

#                                     X_sub = X_train[sampled_idx]
#                                     y_sub = y_train[sampled_idx]
#                                     groups_sub = groups_train[sampled_idx]  # ✅ GroupKFold needs corresponding groups

#                                     gkf = GroupKFold(n_splits=5)

#                                     for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, groups_sub)):
#                                         print(f"[Repeat {repeat+1} | Fold {fold+1}] Train size: {len(tr_idx)} | Val size: {len(val_idx)} | Test size: {len(y_test)}")

#                                         if model_type == "Mamba":
#                                             model = MambaClassifier(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
#                                         elif model_type == "Transformer":
#                                             model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
#                                         else:
#                                             model = RNNClassifier(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)
#                                         optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
#                                         optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
#                                         scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
#                                         criterion = nn.BCEWithLogitsLoss()
#                                         loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size)

#                                         best_loss = float('inf')
#                                         wait = 0
#                                         used_epochs = 0
#                                         for epoch in range(epochs):
#                                             model.train()
#                                             epoch_losses = []
#                                             for xb, yb in loader:
#                                                 xb, yb = xb.to(device), yb.to(device)
#                                                 optimizer.zero_grad()
#                                                 pred = model(xb)
#                                                 loss = criterion(pred, yb)
#                                                 if torch.isnan(loss) or torch.isinf(loss):
#                                                     continue
#                                                 loss.backward()
#                                                 epoch_losses.append(loss.item())
#                                                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#                                                 optimizer.step()
#                                             avg_loss = np.mean(epoch_losses)
#                                             scheduler.step(avg_loss)
#                                             used_epochs += 1
#                                             if avg_loss < best_loss:
#                                                 best_loss = avg_loss
#                                                 wait = 0
#                                             else:
#                                                 wait += 1
#                                                 if wait >= 10:
#                                                     break
#                                         used_epochs_all.append(used_epochs)

#                                         model.eval()
#                                         with torch.no_grad():
#                                             val_input = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
#                                             logits_tensor = model(val_input).cpu().flatten()
#                                             probs = torch.sigmoid(logits_tensor).numpy()
#                                             y_true = y_sub[val_idx].flatten()
#                                             valid_mask = np.isfinite(probs)
#                                             if valid_mask.sum() == 0:
#                                                 continue
#                                             probs = probs[valid_mask]
#                                             y_true = y_true[valid_mask]
#                                             preds = (probs >= 0.5).astype(int)

#                                             all_aucs.append(roc_auc_score(y_true, probs))
#                                             all_auprcs.append(auprc(y_true, probs))
#                                             all_f1s.append(f1_score(y_true, preds))

#                                             if "cv_predictions" not in results:
#                                                 results["cv_predictions"] = []

#                                             results["cv_predictions"].append({
#                                                 "model": model_type,
#                                                 "hidden_dim": hidden_dim,
#                                                 "dropout": dropout,
#                                                 "num_layers": num_layers,
#                                                 "batch_size": batch_size,
#                                                 "lr": lr,
#                                                 "weight_decay": wd,
#                                                 "optimizer": optimizer_name,
#                                                 "repeat": repeat,
#                                                 "fold": fold,
#                                                 "y_true": y_true.tolist(),
#                                                 "y_prob": probs.tolist()
#                                             })

#                                 results["summary"].append({
#                                     "model": model_type,
#                                     "hidden_dim": hidden_dim,
#                                     "dropout": dropout,
#                                     "num_layers": num_layers,
#                                     "batch_size": batch_size,
#                                     "lr": lr,
#                                     "weight_decay": wd,
#                                     "optimizer": optimizer_name,
#                                     "epochs": epochs,
#                                     "AUC_mean": np.mean(all_aucs) if len(all_aucs) == 50 else np.nan,
#                                     "AUC_std": np.std(all_aucs) if len(all_aucs) == 50 else np.nan,
#                                     "AUPRC_mean": np.mean(all_auprcs) if len(all_auprcs) == 50 else np.nan,
#                                     "AUPRC_std": np.std(all_auprcs) if len(all_auprcs) == 50 else np.nan,
#                                     "F1_mean": np.mean(all_f1s) if len(all_f1s) == 50 else np.nan,
#                                     "F1_std": np.std(all_f1s) if len(all_f1s) == 50 else np.nan,
#                                     "used_epochs_mean": np.mean(used_epochs_all) if len(all_aucs) == 50 else np.nan,
#                                     "used_epochs_std": np.std(used_epochs_all) if len(all_aucs) == 50 else np.nan,
#                                 })

# # ========== Save Results ==========
# df_results = pd.DataFrame(results["summary"])
# df_results.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Figs_ClassificationModels/Classification_grid_search_results(Stats)_4.csv", index=False)
# print("✅ Grid search completed and saved.")

# with open ("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Figs_ClassificationModels/Classification_cv_predictions(Stats)_4.json", "w") as f:
#     json.dump(results["cv_predictions"], f)
# print(f"✅ CV predictions saved: {len(results['cv_predictions'])} records.")


Using device: cuda
Training samples before augmentation: 292
Test samples: 73
Training samples after augmentation: 584


Stats Features Only (Classification) -- Test

In [2]:
# ========== Load Best Hyperparameters ==========
results_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(Stats)_5.csv"
df_results = pd.read_csv(results_path)
best_per_model = df_results.sort_values("AUC_mean", ascending=False).groupby("model").first().reset_index()
print("\nBest configuration per model:\n", best_per_model)

# ========== Evaluation ==========
results = []
predictions = []

for _, row in best_per_model.iterrows():
    model_type = row["model"]
    hidden_dim = int(row["hidden_dim"])
    dropout = float(row["dropout"])
    num_layers = int(row["num_layers"])
    epochs = int(row["epochs"])
    batch_size = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr = float(row["lr"])
    input_dim = X_train.shape[2]

    # === Model Selection ===
    if model_type == "Mamba":
        model = MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        model = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        model = RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)
    model = model.to(device)

    optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    criterion = nn.BCEWithLogitsLoss()

    # === Training ===
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                      torch.tensor(y_train, dtype=torch.float32).view(-1, 1)),
        batch_size=batch_size, shuffle=True
    )

    model.train()
    best_loss = float('inf')
    wait = 0
    used_epochs = 0

    for epoch in range(epochs):
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        if len(epoch_losses) == 0:
            continue

        avg_loss = np.mean(epoch_losses)
        scheduler.step(avg_loss)
        used_epochs += 1

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state_dict = model.state_dict()
        else:
            wait += 1
            if wait >= 10:
                print(f"⏹️ Early stopping at epoch {used_epochs} for model {model_type}")
                break

    model.load_state_dict(best_state_dict)

    # === Evaluation on Test Set ===
    model.eval()
    preds = []
    with torch.no_grad():
        test_loader = DataLoader(
            TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                          torch.tensor(y_test, dtype=torch.float32).view(-1, 1)),
            batch_size=batch_size
        )
        for xb, _ in test_loader:
            xb = xb.to(device)
            logits = model(xb).cpu().numpy().reshape(-1)
            probs = 1 / (1 + np.exp(-logits))  # Sigmoid
            preds.append(probs)

    y_pred = np.concatenate(preds)
    y_true = y_test.flatten()
    y_pred_label = (y_pred >= 0.5).astype(int)

    auc_score = roc_auc_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    auprc_score = auc(recall, precision)
    f1 = f1_score(y_true, y_pred_label)

    results.append({
        "model": model_type,
        "AUC_test": auc_score,
        "AUPRC_test": auprc_score,
        "F1_test": f1,
        "used_epochs": used_epochs
    })

    predictions.append({
        "model": model_type,
        "y_true": y_true.tolist(),
        "y_prob": y_pred.tolist()
    })

# ========== Save + Display ==========
results_df = pd.DataFrame(results)
results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_results(Stats)_5.csv", index=False)
print("\n✅ Test performance of best models saved:")
print(results_df)

with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions(Stats)_5.json", "w") as f:
    json.dump(predictions, f)
print("✅ Test predictions saved.")


Best configuration per model:
          model  hidden_dim  dropout  num_layers  batch_size    lr  \
0          GRU          16      0.0           3          16  0.01   
1         LSTM          16      0.0           1          16  0.01   
2        Mamba          64      0.2           3          32  0.01   
3          RNN          32      0.2           2          32  0.01   
4  Transformer          16      0.2           2          16  0.01   

   weight_decay optimizer  epochs  AUC_mean   AUC_std  AUPRC_mean  AUPRC_std  \
0        0.0001       SGD     200  0.805666  0.087675    0.654855   0.126955   
1        0.0001       SGD     200  0.803415  0.062298    0.637585   0.131365   
2        0.0000       SGD     200  0.773547  0.083567    0.611396   0.145179   
3        0.0000       SGD     200  0.799923  0.078550    0.640597   0.136559   
4        0.0001       SGD     200  0.801352  0.069185    0.656128   0.106946   

    F1_mean    F1_std  used_epochs_mean  used_epochs_std  
0  0.564016  

CNN Features (ALL) Only (Classification) -- Training

In [3]:
# === Try more

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from collections import defaultdict
from mamba_ssm import Mamba
import math
import json

# ========== Device ==========
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet3"
)

target_col = "OSI_V2_12th_avg"

tw_features = {tw: [f"f{i}_TW{tw}" for i in range(1, 257)] for tw in range(1, 7)}

X, y, groups = [], [], []

for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target_col]):
        X.append(sequence)
        y.append(1 if row[target_col] >= 7.5 else 0)
        groups.append(row['ResearchID'] if 'ResearchID' in row else idx)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

feature_dim = X.shape[2]

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y, groups, test_size=0.2, random_state=42
)

print(f"Training samples before augmentation: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

print(f"Training samples after augmentation: {len(X_train)}")

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

# ========== Utility ==========
def get_loader(X, y, batch_size):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

def auprc(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3):
# def stratified_sample(y_train, ratio=0.8, pos_ratio=1/3):
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]
    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1
    if len(idx_0) < n0 or len(idx_1) < n1:
        return None, None
    idx = np.concatenate([
        np.random.choice(idx_0, n0, replace=False),
        np.random.choice(idx_1, n1, replace=False)
    ])
    np.random.shuffle(idx)
    return idx, total

# # ========== Train Loop ==========
# epochs = 200
# results = {"cv_predictions": [], "summary": []}
# model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

# for model_type in model_types:
#     for hidden_dim in [16, 32, 64, 128]:
#         for dropout in [0.0, 0.2]:
#             for num_layers in [1, 2, 3]:
#                 for batch_size in [16, 32]:
#                     for lr in [0.001, 0.01]:
#                         for wd in [0, 1e-4]:
#                             for optimizer_name in ["Adam", "SGD"]:
#                                 all_aucs, all_auprcs, all_f1s = [], [], []
#                                 used_epochs_all = []
#                                 for repeat in range(10):
#                                     sampled_idx, _ = stratified_sample(y_train)
#                                     if sampled_idx is None:
#                                         print(f"[{model_type}] Skipping due to not enough positive or negative samples.")
#                                         continue

#                                     X_sub = X_train[sampled_idx]
#                                     y_sub = y_train[sampled_idx]
#                                     groups_sub = groups_train[sampled_idx]  # ✅ GroupKFold needs corresponding groups

#                                     gkf = GroupKFold(n_splits=5)

#                                     for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, groups_sub)):
#                                         print(f"[Repeat {repeat+1} | Fold {fold+1}] Train size: {len(tr_idx)} | Val size: {len(val_idx)} | Test size: {len(y_test)}")

#                                         if model_type == "Mamba":
#                                             model = MambaClassifier(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
#                                         elif model_type == "Transformer":
#                                             model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
#                                         else:
#                                             model = RNNClassifier(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)
#                                         optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
#                                         optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
#                                         scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
#                                         criterion = nn.BCEWithLogitsLoss()
#                                         loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size)

#                                         best_loss = float('inf')
#                                         wait = 0
#                                         used_epochs = 0
#                                         for epoch in range(epochs):
#                                             model.train()
#                                             epoch_losses = []
#                                             for xb, yb in loader:
#                                                 xb, yb = xb.to(device), yb.to(device)
#                                                 optimizer.zero_grad()
#                                                 pred = model(xb)
#                                                 loss = criterion(pred, yb)
#                                                 if torch.isnan(loss) or torch.isinf(loss):
#                                                     continue
#                                                 loss.backward()
#                                                 epoch_losses.append(loss.item())
#                                                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#                                                 optimizer.step()
#                                             avg_loss = np.mean(epoch_losses)
#                                             scheduler.step(avg_loss)
#                                             used_epochs += 1
#                                             if avg_loss < best_loss:
#                                                 best_loss = avg_loss
#                                                 wait = 0
#                                             else:
#                                                 wait += 1
#                                                 if wait >= 10:
#                                                     break
#                                         used_epochs_all.append(used_epochs)

#                                         model.eval()
#                                         with torch.no_grad():
#                                             val_input = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
#                                             logits_tensor = model(val_input).cpu().flatten()
#                                             probs = torch.sigmoid(logits_tensor).numpy()
#                                             y_true = y_sub[val_idx].flatten()
#                                             valid_mask = np.isfinite(probs)
#                                             if valid_mask.sum() == 0:
#                                                 continue
#                                             probs = probs[valid_mask]
#                                             y_true = y_true[valid_mask]
#                                             preds = (probs >= 0.5).astype(int)

#                                             all_aucs.append(roc_auc_score(y_true, probs))
#                                             all_auprcs.append(auprc(y_true, probs))
#                                             all_f1s.append(f1_score(y_true, preds))

#                                             if "cv_predictions" not in results:
#                                                 results["cv_predictions"] = []

#                                             results["cv_predictions"].append({
#                                                 "model": model_type,
#                                                 "hidden_dim": hidden_dim,
#                                                 "dropout": dropout,
#                                                 "num_layers": num_layers,
#                                                 "batch_size": batch_size,
#                                                 "lr": lr,
#                                                 "weight_decay": wd,
#                                                 "optimizer": optimizer_name,
#                                                 "repeat": repeat,
#                                                 "fold": fold,
#                                                 "y_true": y_true.tolist(),
#                                                 "y_prob": probs.tolist()
#                                             })

#                                 results["summary"].append({
#                                     "model": model_type,
#                                     "hidden_dim": hidden_dim,
#                                     "dropout": dropout,
#                                     "num_layers": num_layers,
#                                     "batch_size": batch_size,
#                                     "lr": lr,
#                                     "weight_decay": wd,
#                                     "optimizer": optimizer_name,
#                                     "epochs": epochs,
#                                     "AUC_mean": np.mean(all_aucs) if len(all_aucs) == 50 else np.nan,
#                                     "AUC_std": np.std(all_aucs) if len(all_aucs) == 50 else np.nan,
#                                     "AUPRC_mean": np.mean(all_auprcs) if len(all_auprcs) == 50 else np.nan,
#                                     "AUPRC_std": np.std(all_auprcs) if len(all_auprcs) == 50 else np.nan,
#                                     "F1_mean": np.mean(all_f1s) if len(all_f1s) == 50 else np.nan,
#                                     "F1_std": np.std(all_f1s) if len(all_f1s) == 50 else np.nan,
#                                     "used_epochs_mean": np.mean(used_epochs_all) if len(all_aucs) == 50 else np.nan,
#                                     "used_epochs_std": np.std(used_epochs_all) if len(all_aucs) == 50 else np.nan,
#                                 })

# # ========== Save Results ==========
# df_results = pd.DataFrame(results["summary"])
# df_results.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(CNN)_5.csv", index=False)
# print("✅ Grid search completed and saved.")

# with open ("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_cv_predictions(CNN)_5.json", "w") as f:
#     json.dump(results["cv_predictions"], f)
# print(f"✅ CV predictions saved: {len(results['cv_predictions'])} records.")


Using device: cuda
Training samples before augmentation: 292
Test samples: 73
Training samples after augmentation: 584


CNN Features (ALL) Only (Classification) -- Test

In [4]:
# ========== Load Best Hyperparameters ==========
results_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(CNN)_5.csv"
df_results = pd.read_csv(results_path)
best_per_model = df_results.sort_values("AUC_mean", ascending=False).groupby("model").first().reset_index()
print("\nBest configuration per model:\n", best_per_model)

# ========== Evaluation ==========
results = []
predictions = []

for _, row in best_per_model.iterrows():
    model_type = row["model"]
    hidden_dim = int(row["hidden_dim"])
    dropout = float(row["dropout"])
    num_layers = int(row["num_layers"])
    epochs = int(row["epochs"])
    batch_size = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr = float(row["lr"])
    input_dim = X_train.shape[2]

    # === Model Selection ===
    if model_type == "Mamba":
        model = MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        model = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        model = RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)
    model = model.to(device)

    optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    criterion = nn.BCEWithLogitsLoss()

    # === Training ===
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                      torch.tensor(y_train, dtype=torch.float32).view(-1, 1)),
        batch_size=batch_size, shuffle=True
    )

    model.train()
    best_loss = float('inf')
    wait = 0
    used_epochs = 0

    for epoch in range(epochs):
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        if len(epoch_losses) == 0:
            continue

        avg_loss = np.mean(epoch_losses)
        scheduler.step(avg_loss)
        used_epochs += 1

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state_dict = model.state_dict()
        else:
            wait += 1
            if wait >= 10:
                print(f"⏹️ Early stopping at epoch {used_epochs} for model {model_type}")
                break

    model.load_state_dict(best_state_dict)

    # === Evaluation on Test Set ===
    model.eval()
    preds = []
    with torch.no_grad():
        test_loader = DataLoader(
            TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                          torch.tensor(y_test, dtype=torch.float32).view(-1, 1)),
            batch_size=batch_size
        )
        for xb, _ in test_loader:
            xb = xb.to(device)
            logits = model(xb).cpu().numpy().reshape(-1)
            probs = 1 / (1 + np.exp(-logits))  # Sigmoid
            preds.append(probs)

    y_pred = np.concatenate(preds)
    y_true = y_test.flatten()
    y_pred_label = (y_pred >= 0.5).astype(int)

    auc_score = roc_auc_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    auprc_score = auc(recall, precision)
    f1 = f1_score(y_true, y_pred_label)

    results.append({
        "model": model_type,
        "AUC_test": auc_score,
        "AUPRC_test": auprc_score,
        "F1_test": f1,
        "used_epochs": used_epochs
    })

    predictions.append({
        "model": model_type,
        "y_true": y_true.tolist(),
        "y_prob": y_pred.tolist()
    })

# ========== Save + Display ==========
results_df = pd.DataFrame(results)
results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_results(CNN)_5.csv", index=False)
print("\n✅ Test performance of best models saved:")
print(results_df)

with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions(CNN)_5.json", "w") as f:
    json.dump(predictions, f)
print("✅ Test predictions saved.")


Best configuration per model:
          model  hidden_dim  dropout  num_layers  batch_size     lr  \
0          GRU          16      0.2           3          32  0.010   
1         LSTM          32      0.2           2          16  0.010   
2        Mamba          16      0.0           1          16  0.001   
3          RNN         128      0.0           3          16  0.001   
4  Transformer          16      0.2           3          16  0.010   

   weight_decay optimizer  epochs  AUC_mean   AUC_std  AUPRC_mean  AUPRC_std  \
0        0.0000       SGD     200  0.778610  0.074304    0.605635   0.155548   
1        0.0000       SGD     200  0.772417  0.069005    0.611419   0.126891   
2        0.0001       SGD     200  0.748230  0.073960    0.577631   0.128605   
3        0.0000       SGD     200  0.777890  0.078512    0.619065   0.150510   
4        0.0001       SGD     200  0.769303  0.067106    0.570562   0.125373   

    F1_mean    F1_std  used_epochs_mean  used_epochs_std  
0  0.42

Stats + CNN Features (Classification) -- Training

In [5]:
# === Try more

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from collections import defaultdict
from mamba_ssm import Mamba
import math
import json

# ========== Device ==========
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet3"
)

target_col = "OSI_V2_12th_avg"

base_template = ["OSI_mean_TW{}", "OSI_std_TW{}"]

tw_features = {
    tw: [f.format(tw) for f in base_template] + [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

X, y, groups = [], [], []

for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target_col]):
        X.append(sequence)
        y.append(1 if row[target_col] >= 7.5 else 0)
        groups.append(row['ResearchID'] if 'ResearchID' in row else idx)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

feature_dim = X.shape[2]

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y, groups, test_size=0.2, random_state=42
)

print(f"Training samples before augmentation: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

print(f"Training samples after augmentation: {len(X_train)}")

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

# ========== Utility ==========
def get_loader(X, y, batch_size):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

def auprc(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3):
# def stratified_sample(y_train, ratio=0.8, pos_ratio=1/3):
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]
    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1
    if len(idx_0) < n0 or len(idx_1) < n1:
        return None, None
    idx = np.concatenate([
        np.random.choice(idx_0, n0, replace=False),
        np.random.choice(idx_1, n1, replace=False)
    ])
    np.random.shuffle(idx)
    return idx, total

# # ========== Train Loop ==========
# epochs = 200
# results = {"cv_predictions": [], "summary": []}
# model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

# for model_type in model_types:
#     for hidden_dim in [16, 32, 64, 128]:
#         for dropout in [0.0, 0.2]:
#             for num_layers in [1, 2, 3]:
#                 for batch_size in [16, 32]:
#                     for lr in [0.001, 0.01]:
#                         for wd in [0, 1e-4]:
#                             for optimizer_name in ["Adam", "SGD"]:
#                                 all_aucs, all_auprcs, all_f1s = [], [], []
#                                 used_epochs_all = []
#                                 for repeat in range(10):
#                                     sampled_idx, _ = stratified_sample(y_train)
#                                     if sampled_idx is None:
#                                         print(f"[{model_type}] Skipping due to not enough positive or negative samples.")
#                                         continue

#                                     X_sub = X_train[sampled_idx]
#                                     y_sub = y_train[sampled_idx]
#                                     groups_sub = groups_train[sampled_idx]  # ✅ GroupKFold needs corresponding groups

#                                     gkf = GroupKFold(n_splits=5)

#                                     for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, groups_sub)):
#                                         print(f"[Repeat {repeat+1} | Fold {fold+1}] Train size: {len(tr_idx)} | Val size: {len(val_idx)} | Test size: {len(y_test)}")

#                                         if model_type == "Mamba":
#                                             model = MambaClassifier(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
#                                         elif model_type == "Transformer":
#                                             model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
#                                         else:
#                                             model = RNNClassifier(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)
#                                         optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
#                                         optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
#                                         scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
#                                         criterion = nn.BCEWithLogitsLoss()
#                                         loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size)

#                                         best_loss = float('inf')
#                                         wait = 0
#                                         used_epochs = 0
#                                         for epoch in range(epochs):
#                                             model.train()
#                                             epoch_losses = []
#                                             for xb, yb in loader:
#                                                 xb, yb = xb.to(device), yb.to(device)
#                                                 optimizer.zero_grad()
#                                                 pred = model(xb)
#                                                 loss = criterion(pred, yb)
#                                                 if torch.isnan(loss) or torch.isinf(loss):
#                                                     continue
#                                                 loss.backward()
#                                                 epoch_losses.append(loss.item())
#                                                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#                                                 optimizer.step()
#                                             avg_loss = np.mean(epoch_losses)
#                                             scheduler.step(avg_loss)
#                                             used_epochs += 1
#                                             if avg_loss < best_loss:
#                                                 best_loss = avg_loss
#                                                 wait = 0
#                                             else:
#                                                 wait += 1
#                                                 if wait >= 10:
#                                                     break
#                                         used_epochs_all.append(used_epochs)

#                                         model.eval()
#                                         with torch.no_grad():
#                                             val_input = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
#                                             logits_tensor = model(val_input).cpu().flatten()
#                                             probs = torch.sigmoid(logits_tensor).numpy()
#                                             y_true = y_sub[val_idx].flatten()
#                                             valid_mask = np.isfinite(probs)
#                                             if valid_mask.sum() == 0:
#                                                 continue
#                                             probs = probs[valid_mask]
#                                             y_true = y_true[valid_mask]
#                                             preds = (probs >= 0.5).astype(int)

#                                             all_aucs.append(roc_auc_score(y_true, probs))
#                                             all_auprcs.append(auprc(y_true, probs))
#                                             all_f1s.append(f1_score(y_true, preds))

#                                             if "cv_predictions" not in results:
#                                                 results["cv_predictions"] = []

#                                             results["cv_predictions"].append({
#                                                 "model": model_type,
#                                                 "hidden_dim": hidden_dim,
#                                                 "dropout": dropout,
#                                                 "num_layers": num_layers,
#                                                 "batch_size": batch_size,
#                                                 "lr": lr,
#                                                 "weight_decay": wd,
#                                                 "optimizer": optimizer_name,
#                                                 "repeat": repeat,
#                                                 "fold": fold,
#                                                 "y_true": y_true.tolist(),
#                                                 "y_prob": probs.tolist()
#                                             })

#                                 results["summary"].append({
#                                     "model": model_type,
#                                     "hidden_dim": hidden_dim,
#                                     "dropout": dropout,
#                                     "num_layers": num_layers,
#                                     "batch_size": batch_size,
#                                     "lr": lr,
#                                     "weight_decay": wd,
#                                     "optimizer": optimizer_name,
#                                     "epochs": epochs,
#                                     "AUC_mean": np.mean(all_aucs) if len(all_aucs) == 50 else np.nan,
#                                     "AUC_std": np.std(all_aucs) if len(all_aucs) == 50 else np.nan,
#                                     "AUPRC_mean": np.mean(all_auprcs) if len(all_auprcs) == 50 else np.nan,
#                                     "AUPRC_std": np.std(all_auprcs) if len(all_auprcs) == 50 else np.nan,
#                                     "F1_mean": np.mean(all_f1s) if len(all_f1s) == 50 else np.nan,
#                                     "F1_std": np.std(all_f1s) if len(all_f1s) == 50 else np.nan,
#                                     "used_epochs_mean": np.mean(used_epochs_all) if len(all_aucs) == 50 else np.nan,
#                                     "used_epochs_std": np.std(used_epochs_all) if len(all_aucs) == 50 else np.nan,
#                                 })

# # ========== Save Results ==========
# df_results = pd.DataFrame(results["summary"])
# df_results.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(Stats_CNN)_5.csv", index=False)
# print("✅ Grid search completed and saved.")

# with open ("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_cv_predictions(Stats_CNN)_5.json", "w") as f:
#     json.dump(results["cv_predictions"], f)
# print(f"✅ CV predictions saved: {len(results['cv_predictions'])} records.")


Using device: cuda
Training samples before augmentation: 284
Test samples: 71
Training samples after augmentation: 568


Stats + CNN Features (Classification) -- Test

In [6]:
# ========== Load Best Hyperparameters ==========
results_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(Stats_CNN)_5.csv"
df_results = pd.read_csv(results_path)
best_per_model = df_results.sort_values("AUC_mean", ascending=False).groupby("model").first().reset_index()
print("\nBest configuration per model:\n", best_per_model)

# ========== Evaluation ==========
results = []
predictions = []

for _, row in best_per_model.iterrows():
    model_type = row["model"]
    hidden_dim = int(row["hidden_dim"])
    dropout = float(row["dropout"])
    num_layers = int(row["num_layers"])
    epochs = int(row["epochs"])
    batch_size = int(row["batch_size"])
    optimizer_name = row["optimizer"]
    lr = float(row["lr"])
    input_dim = X_train.shape[2]

    # === Model Selection ===
    if model_type == "Mamba":
        model = MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        model = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        model = RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)
    model = model.to(device)

    optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    criterion = nn.BCEWithLogitsLoss()

    # === Training ===
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                      torch.tensor(y_train, dtype=torch.float32).view(-1, 1)),
        batch_size=batch_size, shuffle=True
    )

    model.train()
    best_loss = float('inf')
    wait = 0
    used_epochs = 0

    for epoch in range(epochs):
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        if len(epoch_losses) == 0:
            continue

        avg_loss = np.mean(epoch_losses)
        scheduler.step(avg_loss)
        used_epochs += 1

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state_dict = model.state_dict()
        else:
            wait += 1
            if wait >= 10:
                print(f"⏹️ Early stopping at epoch {used_epochs} for model {model_type}")
                break

    model.load_state_dict(best_state_dict)

    # === Evaluation on Test Set ===
    model.eval()
    preds = []
    with torch.no_grad():
        test_loader = DataLoader(
            TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                          torch.tensor(y_test, dtype=torch.float32).view(-1, 1)),
            batch_size=batch_size
        )
        for xb, _ in test_loader:
            xb = xb.to(device)
            logits = model(xb).cpu().numpy().reshape(-1)
            probs = 1 / (1 + np.exp(-logits))  # Sigmoid
            preds.append(probs)

    y_pred = np.concatenate(preds)
    y_true = y_test.flatten()
    y_pred_label = (y_pred >= 0.5).astype(int)

    auc_score = roc_auc_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    auprc_score = auc(recall, precision)
    f1 = f1_score(y_true, y_pred_label)

    results.append({
        "model": model_type,
        "AUC_test": auc_score,
        "AUPRC_test": auprc_score,
        "F1_test": f1,
        "used_epochs": used_epochs
    })

    predictions.append({
        "model": model_type,
        "y_true": y_true.tolist(),
        "y_prob": y_pred.tolist()
    })

# ========== Save + Display ==========
results_df = pd.DataFrame(results)
results_df.to_csv("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_results(Stats_CNN)_5.csv", index=False)
print("\n✅ Test performance of best models saved:")
print(results_df)

with open("/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions(Stats_CNN)_5.json", "w") as f:
    json.dump(predictions, f)
print("✅ Test predictions saved.")


Best configuration per model:
          model  hidden_dim  dropout  num_layers  batch_size     lr  \
0          GRU         128      0.2           3          32  0.010   
1         LSTM          32      0.2           2          16  0.010   
2        Mamba         128      0.0           1          16  0.001   
3          RNN         128      0.2           3          32  0.010   
4  Transformer          16      0.2           2          16  0.010   

   weight_decay optimizer  epochs  AUC_mean   AUC_std  AUPRC_mean  AUPRC_std  \
0        0.0000       SGD     200  0.777941  0.066103    0.603472   0.142746   
1        0.0000       SGD     200  0.785045  0.080173    0.616234   0.134917   
2        0.0001       SGD     200  0.747633  0.067099    0.571893   0.138562   
3        0.0000       SGD     200  0.785718  0.059201    0.624109   0.139475   
4        0.0000       SGD     200  0.773259  0.078060    0.611194   0.134246   

    F1_mean    F1_std  used_epochs_mean  used_epochs_std  
0  0.54